# D2C Customer Churn: Data Audit, EDA & Business Understanding

This notebook audits every raw file, checks joins and leakage risks, performs EDA, and builds dataset-backed churn-risk hypotheses.

In [1]:
import pandas as pd
from pathlib import Path
SNAPSHOT = pd.Timestamp('2025-09-30')
DATA = Path('data/raw')
customers = pd.read_csv(DATA/'customers.csv', parse_dates=['signup_date'])
orders = pd.read_csv(DATA/'orders.csv', parse_dates=['order_date'])
support = pd.read_csv(DATA/'support_tickets.csv', parse_dates=['ticket_date'])
web = pd.read_csv(DATA/'web_events_snapshot.csv', parse_dates=['snapshot_date'])
labels = pd.read_csv(DATA/'churn_labels.csv', parse_dates=['snapshot_date'])
interventions = pd.read_csv(DATA/'intervention_history.csv', parse_dates=['snapshot_date'])
rfm = pd.read_csv(DATA/'rfm_modeling_snapshot.csv', parse_dates=['snapshot_date'])
frames = {'customers': customers, 'orders': orders, 'support': support, 'web': web, 'labels': labels, 'interventions': interventions, 'rfm': rfm}
[(name, df.shape) for name, df in frames.items()]

[('customers', (2400, 9)),
 ('orders', (10009, 10)),
 ('support', (1921, 8)),
 ('web', (2400, 10)),
 ('labels', (2400, 4)),
 ('interventions', (2400, 5)),
 ('rfm', (2400, 29))]

## Schema, Missingness, and Join Checks

In [2]:
pd.read_csv('outputs/dataset_inventory.csv')

                 dataset   rows  columns  customer_ids  duplicate_rows  \
0              customers   2400        9          2400               0   
1                 orders  10009       10          2400               0   
2        support_tickets   1921        8          1247               0   
3    web_events_snapshot   2400       10          2400               0   
4           churn_labels   2400        4          2400               0   
5   intervention_history   2400        5          2400               0   
6  rfm_modeling_snapshot   2400       29          2400               0   

   missing_cells  unknown_customer_ids    date_min    date_max  
0           1787                     0  2024-01-01  2025-09-15  
1             80                     0  2024-01-09  2025-11-29  
2              0                     0  2024-01-13  2025-09-30  
3              0                     0  2025-09-30  2025-09-30  
4              0                     0  2025-09-30  2025-09-30  
5              0 

In [3]:
pd.read_csv('outputs/missing_values.csv')

                 dataset        column  missing_count  missing_rate
0              customers  loyalty_tier           1386      0.577500
1  rfm_modeling_snapshot  loyalty_tier           1386      0.577500
2              customers     skin_type            401      0.167083
3                 orders        rating             80      0.007993

In [4]:
pd.read_csv('outputs/join_key_audit.csv')

                 dataset   rows  unique_customer_ids  \
0                 orders  10009                 2400   
1        support_tickets   1921                 1247   
2    web_events_snapshot   2400                 2400   
3           churn_labels   2400                 2400   
4   intervention_history   2400                 2400   
5  rfm_modeling_snapshot   2400                 2400   

   customers_missing_from_file  unknown_customer_id_rows  
0                            0                         0  
1                         1153                         0  
2                            0                         0  
3                            0                         0  
4                            0                         0  
5                            0                         0  

## Invalid Values, Duplicate-Like Orders, Outliers, and Leakage

In [5]:
pd.read_csv('outputs/invalid_value_checks.csv')

                 dataset                           column  invalid_values  \
0                 orders             quantity_outside_1_4             NaN   
1                 orders         discount_outside_0_70pct             NaN   
2                 orders       delivery_days_outside_1_11             NaN   
3                 orders              returned_not_binary             NaN   
4                 orders      rating_outside_1_5_non_null             NaN   
5        support_tickets        resolution_hours_negative             NaN   
6        support_tickets           sentiment_outside_-1_1             NaN   
7        support_tickets              reopened_not_binary             NaN   
8           churn_labels                target_not_binary             NaN   
9   intervention_history  none_campaign_with_nonzero_cost             NaN   
10   web_events_snapshot         negative_activity_counts             NaN   

    invalid_count  
0               0  
1               0  
2              

In [6]:
pd.read_csv('outputs/duplicate_like_orders.csv').head(20)

         order_id customer_id  order_date   category  quantity  gross_amount  \
0   ORD008249_DUP   CUST00153  2025-11-04  Hair Care         1        321.31   
1   ORD002124_DUP   CUST00628  2025-03-18  Skin Care         1        410.04   
2   ORD002862_DUP   CUST00837  2025-07-12  Hair Care         3        952.02   
3   ORD002916_DUP   CUST00848  2025-09-26  Skin Care         1        547.18   
4   ORD002970_DUP   CUST00869  2024-12-22  Fragrance         1        818.64   
5   ORD008836_DUP   CUST00875  2025-10-23  Baby Care         2        711.20   
6   ORD003897_DUP   CUST01140  2025-04-14  Baby Care         2        769.96   
7   ORD004577_DUP   CUST01335  2025-02-12   Wellness         1        533.07   
8   ORD005451_DUP   CUST01601  2024-11-07     Makeup         2       1160.41   
9   ORD005529_DUP   CUST01621  2024-08-12  Baby Care         1        339.33   
10  ORD006220_DUP   CUST01820  2025-01-22     Makeup         1        562.71   
11  ORD009518_DUP   CUST01820  2025-10-1

In [7]:
pd.read_csv('outputs/top_order_value_outliers.csv').head(15)

     order_id customer_id  order_date   category  quantity  gross_amount  \
0   ORD006374   CUST01868  2025-03-29  Skin Care         3      24789.38   
1   ORD000701   CUST00211  2024-11-27  Fragrance         2      22719.45   
2   ORD007206   CUST02106  2024-07-13  Fragrance         2      15957.48   
3   ORD009649   CUST01988  2025-10-25  Fragrance         1      12312.12   
4   ORD004428   CUST01295  2025-05-01  Baby Care         2      10643.82   
5   ORD004650   CUST01360  2024-10-09  Fragrance         2       8777.20   
6   ORD005399   CUST01584  2024-12-31  Fragrance         1       8022.50   
7   ORD007765   CUST02287  2025-06-22  Fragrance         4       3746.76   
8   ORD000500   CUST00159  2024-06-13  Fragrance         4       3376.32   
9   ORD001120   CUST00324  2024-12-30  Fragrance         4       3341.27   
10  ORD004146   CUST01214  2025-02-09  Fragrance         3       3098.18   
11  ORD004910   CUST01439  2025-09-30  Fragrance         4       3032.67   
12  ORD00473

In [8]:
pre_orders = orders[orders.order_date <= SNAPSHOT]
post_orders = orders[orders.order_date > SNAPSHOT]
{'pre_snapshot_orders': len(pre_orders), 'post_snapshot_orders_label_window_only': len(post_orders)}

{'pre_snapshot_orders': 8137, 'post_snapshot_orders_label_window_only': 1872}

## Customer-Level EDA Master Table

The master table joins customers to labels, pre-snapshot order features, support features, web/app snapshot metrics, and intervention history.

In [9]:
master = pd.read_csv('outputs/customer_eda_master.csv')
master.head()

  customer_id signup_date city_tier age_group acquisition_channel  \
0   CUST00001  2024-04-24    Tier 1     18-24           Instagram   
1   CUST00002  2025-06-01    Tier 2     25-34         Marketplace   
2   CUST00003  2025-03-08    Tier 1     25-34          Influencer   
3   CUST00004  2025-04-15    Tier 3     25-34       Google Search   
4   CUST00005  2024-08-21    Tier 3     35-44             Organic   

  loyalty_tier preferred_category    skin_type marketing_consent  \
0       Silver             Makeup       Normal               Yes   
1       Silver          Hair Care  Combination               Yes   
2          NaN          Skin Care         Oily               Yes   
3          NaN          Fragrance       Normal                No   
4         Gold          Hair Care  Combination               Yes   

  snapshot_date  ...  email_opens_30d campaign_clicks_30d  \
0    2025-09-30  ...                2                   0   
1    2025-09-30  ...                0                 

## Churn Distribution and Profile EDA

In [10]:
pd.read_csv('outputs/churn_distribution.csv')

   churn_next_60d  customers
0               0       1273
1               1       1127

In [11]:
pd.read_csv('outputs/profile_churn_city_tier.csv')

  city_tier  customers  churn_rate
0    Tier 2        870    0.477011
1    Tier 1       1005    0.473632
2    Tier 3        525    0.449524

In [12]:
pd.read_csv('outputs/profile_churn_age_group.csv')

  age_group  customers  churn_rate
0     35-44        534    0.483146
1     25-34       1045    0.471770
2       45+        261    0.463602
3     18-24        560    0.455357

In [13]:
pd.read_csv('outputs/profile_churn_acquisition.csv')

  acquisition_channel  customers  churn_rate
0       Google Search        466    0.504292
1           Instagram        517    0.499033
2         Marketplace        456    0.491228
3          Influencer        231    0.476190
4            Referral        396    0.421717
5             Organic        334    0.398204

In [14]:
pd.read_csv('outputs/profile_churn_loyalty.csv')

  loyalty_status  customers  churn_rate
0         Silver        590    0.488136
1   Not enrolled       1386    0.483405
2           Gold        319    0.407524
3       Platinum        105    0.371429

## Order, Monetary, Return, and Support EDA

In [15]:
pd.read_csv('outputs/order_recency_churn.csv')

  recency_band  customers  churn_rate
0         0-30        699    0.117310
1        31-60        441    0.303855
2        61-90        361    0.457064
3       91-120        234    0.653846
4      121-180        363    0.873278
5         181+        302    0.913907

In [16]:
pd.read_csv('outputs/order_frequency_churn.csv')

  frequency_band  customers  churn_rate
0            0-1        696    0.502874
1              2        380    0.565789
2            3-4        659    0.490137
3            5-8        580    0.370690
4             9+         85    0.282353

In [17]:
pd.read_csv('outputs/monetary_churn.csv')

  monetary_band  customers  avg_spend  churn_rate
0        lowest        480   0.539583    0.539583
1           low        480   0.535417    0.535417
2        middle        480   0.481250    0.481250
3          high        480   0.445833    0.445833
4       highest        480   0.345833    0.345833

In [18]:
pd.read_csv('outputs/return_rate_churn.csv')

  return_band  customers  churn_rate
0           0       1932    0.469979
1       0-20%        199    0.341709
2      20-50%        209    0.507177
3        50%+         60    0.750000

In [19]:
pd.read_csv('outputs/support_ticket_churn.csv')

  ticket_band  customers  churn_rate
0           0       1153    0.471813
1           1        781    0.503201
2           2        314    0.436306
3          3+        152    0.348684

In [20]:
pd.read_csv('outputs/support_issue_churn.csv')

         issue_type  tickets  churn_rate  avg_sentiment  avg_resolution_hours  \
0  product_reaction      194    0.448454      -0.597680             36.536598   
1        wrong_item      213    0.446009      -0.407840             20.904695   
2     payment_issue      191    0.439791      -0.348796             20.144503   
3     general_query      324    0.435185      -0.415062             20.830247   
4      damaged_item      277    0.433213      -0.307942             19.184116   
5      refund_delay      345    0.428986      -0.652319             36.459420   
6     late_delivery      377    0.427056      -0.351326             20.133422   

   reopened_rate  
0       0.268041  
1       0.126761  
2       0.157068  
3       0.163580  
4       0.151625  
5       0.223188  
6       0.156499  

## Web/App and Campaign EDA

In [21]:
pd.read_csv('outputs/web_last_visit_churn.csv')

  visit_band  customers  churn_rate
0        0-3        528    0.160985
1        4-7        295    0.233898
2       8-14        401    0.339152
3      15-30        695    0.592806
4        31+        481    0.883576

In [22]:
pd.read_csv('outputs/web_sessions_churn.csv')

  session_band  customers  churn_rate
0            0        190    0.663158
1          1-2        592    0.638514
2          3-5        599    0.509182
3         6-10        674    0.364985
4          11+        345    0.208696

In [23]:
pd.read_csv('outputs/campaign_churn.csv')

  last_campaign_received  customers  avg_campaign_cost  churn_rate
0             new_launch        498           0.510040    0.510040
1        bundle_discount        473           0.469345    0.469345
2          free_shipping        469           0.462687    0.462687
3          welcome_offer        453           0.452539    0.452539
4                   none        507           0.451677    0.451677

In [24]:
pd.read_csv('outputs/priority_bucket_churn.csv')

  manual_priority_bucket  customers  churn_rate
0                   high       1163    0.747206
1                 medium        749    0.279039
2                    low        488    0.100410

## Churn-Risk Hypotheses

In [25]:
pd.read_csv('outputs/churn_risk_hypotheses.csv')

                                          hypothesis  \
0  Stale order recency is the clearest churn warn...   
1  Recent web/app inactivity is a strong disengag...   
2  Support tickets need issue-level triage rather...   
3  Return-heavy customers need product or fulfill...   
4  Campaign history reflects risk state and needs...   
5  Loyalty status is associated with different ch...   

                                      evidence_table  \
0                            order_recency_churn.csv   
1                           web_last_visit_churn.csv   
2  support_ticket_churn.csv / support_issue_churn...   
3                              return_rate_churn.csv   
4                                 campaign_churn.csv   
5                          profile_churn_loyalty.csv   

                                            evidence  \
0  Customers in the 181+ day recency band churn a...   
1  The 31+ days-since-visit band churns at 88.4%,...   
2  One-ticket customers churn at 50.3%; the hi

The charts saved under `outputs/` support the same hypotheses: recency, last visit, support tickets, return rate, campaign history, and acquisition channel.